In [1]:
!pip install pandas

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import os
import pandas as pd
import numpy as np
#import networkx as nx
#import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
hidden_units = [32, 32]
learning_rate = 0.01
dropout_rate = 0.5
num_epochs = 2
batch_size = 8 #256


In [4]:
def compile_model(model):
    # Compile the model.
    model.compile(
        optimizer="rmsprop",
        #optimizer=keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        # Tati: categorical_crossentropy, expects the labels to follow a categorical encoding. 
        #       With integer labels, you should use sparse_categorical_crossentropy.
        #       This new loss function is still mathematically the same as categorical_crossentropy; it just has a different interface.
        #metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )
    # Create an early stopping callback.
    #early_stopping = keras.callbacks.EarlyStopping(
    #    monitor="val_acc", patience=50, restore_best_weights=True
    #
    #)
    return model

# This function trains an input model using the given training data.
def run_experiment(model, x_train, y_train): 
    guardarModelo = keras.callbacks.ModelCheckpoint(
        filepath="/mnt/modelos1/",  # "checkpoint_path.keras",
        monitor="val_loss",
        save_best_only=True,
        save_format="tf",
    )
    # Fit the model.
    history = model.fit(
        x=x_train,
        y=y_train,
        epochs=num_epochs,
        batch_size=batch_size,
        validation_split=0.15,
        callbacks=[guardarModelo], #early_stopping],
    )

    return history


def create_ffn(hidden_units, dropout_rate, name=None):
    fnn_layers = []

    for units in hidden_units:
        fnn_layers.append(layers.BatchNormalization())
        fnn_layers.append(layers.Dropout(dropout_rate))
        fnn_layers.append(layers.Dense(units, activation=tf.nn.gelu))

    return keras.Sequential(fnn_layers, name=name)


In [5]:
class GraphConvLayer(layers.Layer):
    def __init__(
        self,
        hidden_units,
        dropout_rate=0.2,
        aggregation_type="mean",
        combination_type="concat",
        normalize=False,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.aggregation_type = aggregation_type
        self.combination_type = combination_type
        self.normalize = normalize

        self.ffn_prepare = create_ffn(hidden_units, dropout_rate)
        if self.combination_type == "gated":
            self.update_fn = layers.GRU(
                units=hidden_units,
                activation="tanh",
                recurrent_activation="sigmoid",
                dropout=dropout_rate,
                return_state=True,
                recurrent_dropout=dropout_rate,
            )
        else:
            self.update_fn = create_ffn(hidden_units, dropout_rate)

    def prepare(self, node_repesentations, weights=None):
        # node_repesentations shape is [num_edges, embedding_dim].
        messages = self.ffn_prepare(node_repesentations)
        if weights is not None:
            messages = messages * tf.expand_dims(weights, -1)
        return messages

    def aggregate(self, node_indices, neighbour_messages, node_repesentations):
        # node_indices shape is [num_edges].
        # neighbour_messages shape: [num_edges, representation_dim].
        # node_repesentations shape is [num_nodes, representation_dim]
        num_nodes = node_repesentations.shape[0]
        if self.aggregation_type == "sum":
            aggregated_message = tf.math.unsorted_segment_sum(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        elif self.aggregation_type == "mean":
            aggregated_message = tf.math.unsorted_segment_mean(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        elif self.aggregation_type == "max":
            aggregated_message = tf.math.unsorted_segment_max(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        else:
            raise ValueError(f"Invalid aggregation type: {self.aggregation_type}.")

        return aggregated_message

    def update(self, node_repesentations, aggregated_messages):
        # node_repesentations shape is [num_nodes, representation_dim].
        # aggregated_messages shape is [num_nodes, representation_dim].
        if self.combination_type == "gru":
            # Create a sequence of two elements for the GRU layer.
            h = tf.stack([node_repesentations, aggregated_messages], axis=1)
        elif self.combination_type == "concat":
            # Concatenate the node_repesentations and aggregated_messages.
            h = tf.concat([node_repesentations, aggregated_messages], axis=1)
        elif self.combination_type == "add":
            # Add node_repesentations and aggregated_messages.
            h = node_repesentations + aggregated_messages
        else:
            raise ValueError(f"Invalid combination type: {self.combination_type}.")

        # Apply the processing function.
        node_embeddings = self.update_fn(h)
        if self.combination_type == "gru":
            node_embeddings = tf.unstack(node_embeddings, axis=1)[-1]

        if self.normalize:
            node_embeddings = tf.nn.l2_normalize(node_embeddings, axis=-1)
        return node_embeddings

    def call(self, inputs):
        """Process the inputs to produce the node_embeddings.

        inputs: a tuple of three elements: node_repesentations, edges, edge_weights.
        Returns: node_embeddings of shape [num_nodes, representation_dim].
        """

        node_repesentations, edges, edge_weights = inputs
        # Get node_indices (source) and neighbour_indices (target) from edges.
        node_indices, neighbour_indices = edges[0], edges[1]
        # neighbour_repesentations shape is [num_edges, representation_dim].
        neighbour_repesentations = tf.gather(node_repesentations, neighbour_indices)

        # Prepare the messages of the neighbours.
        neighbour_messages = self.prepare(neighbour_repesentations, edge_weights)
        # Aggregate the neighbour messages.
        aggregated_messages = self.aggregate(
            node_indices, neighbour_messages, node_repesentations
        )
        # Update the node embedding with the neighbour messages.
        return self.update(node_repesentations, aggregated_messages)


In [6]:
class GNNNodeClassifier(tf.keras.Model):
    def __init__(
        self,
        graph_info,
        num_classes,
        hidden_units,
        aggregation_type="sum",
        combination_type="concat",
        dropout_rate=0.2,
        normalize=True,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        # Unpack graph_info to three elements: node_features, edges, and edge_weight.
        node_features, edges, edge_weights = graph_info
        self.node_features = node_features
        self.edges = edges
        self.edge_weights = edge_weights
        # Set edge_weights to ones if not provided.
        if self.edge_weights is None:
            self.edge_weights = tf.ones(shape=edges.shape[1])
        # Scale edge_weights to sum to 1.
        self.edge_weights = self.edge_weights / tf.math.reduce_sum(self.edge_weights)

        # Create a process layer.
        self.preprocess = create_ffn(hidden_units, dropout_rate, name="preprocess")
        # Create the first GraphConv layer.
        self.conv1 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv1",
        )
        # Create the second GraphConv layer.
        self.conv2 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv2",
        )
        # Create a postprocess layer.
        self.postprocess = create_ffn(hidden_units, dropout_rate, name="postprocess")
        # Create a compute logits layer.
        self.compute_logits = layers.Dense(units=num_classes, name="logits")  
          # Tati: For Tensorflow: logits is a name that it is thought to imply that this Tensor is the quantity that is being mapped to probabilities by the Softmax

    def call(self, input_node_indices):
        # Preprocess the node_features to produce node representations.
        x = self.preprocess(self.node_features)
        # Apply the first graph conv layer.
        x1 = self.conv1((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x1 + x
        # Apply the second graph conv layer.
        x2 = self.conv2((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x2 + x
        # Postprocess node embedding.
        x = self.postprocess(x)
        # Fetch node embeddings for the input node_indices.
        node_embeddings = tf.gather(x, input_node_indices)
        # Compute logits
        return self.compute_logits(node_embeddings)


In [7]:
# Training
training_grafos = pd.read_csv(
    "/mnt/training_GRAFOS.pkts.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

training_features_tmp = pd.read_csv(
    "/mnt/training_FEATURES.pkts.SINnorm.csv",
    sep=",",  
    header=0
)


# Validation
validation_grafos = pd.read_csv(
    "/mnt/validation_GRAFOS.pkts.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

validation_features_tmp = pd.read_csv(
    "/mnt/validation_FEATURES.pkts.SINnorm.csv",
    sep=",",  
    header=0
)



class_values = sorted(training_features_tmp["label"].unique())
class_idx = {name: id for id, name in enumerate(class_values)}
node_idx_train = {name: idx for idx, name in enumerate(sorted(training_features_tmp["node"].unique()))}
node_idx_val = {name: idx for idx, name in enumerate(sorted(validation_features_tmp["node"].unique()))}

training_grafos["source"] = training_grafos["source"].apply(lambda name: node_idx_train[name])
training_grafos["target"] = training_grafos["target"].apply(lambda name: node_idx_train[name])

training_features = training_features_tmp.loc[:,["node","ID","OD","IDW","ODW","label"]].copy()
training_features["node"] = training_features_tmp["node"].apply(lambda name: node_idx_train[name])
training_features["label"] = training_features_tmp["label"].apply(lambda value: class_idx[value])

validation_grafos["source"] = validation_grafos["source"].apply(lambda name: node_idx_val[name])
validation_grafos["target"] = validation_grafos["target"].apply(lambda name: node_idx_val[name])

validation_features = validation_features_tmp.loc[:,["node","ID","OD","IDW","ODW","label"]].copy()
validation_features["node"] = validation_features_tmp["node"].apply(lambda name: node_idx_val[name])
validation_features["label"] = validation_features_tmp["label"].apply(lambda value: class_idx[value])


# Create an edges array (adjacency matrix) of shape [2, num_edges]
training_edges = training_grafos[["source", "target"]].to_numpy().T
validation_edges = validation_grafos[["source", "target"]].to_numpy().T

# Create an edge weights array.
training_edge_weights = training_grafos[["weight"]].to_numpy().T 
training_edge_weights = training_edge_weights.reshape((training_edges.shape[-1],))
training_edge_weights = tf.convert_to_tensor(training_edge_weights)

# Create a node features array of shape [num_nodes, num_features].
feature_names = set(training_features.columns) - {"node", "label"}
training_node_features = tf.cast(
    training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32
)

# Create graph info tuple with node_features, edges, and edge_weights.
graph_info = (training_node_features, training_edges, training_edge_weights)

print("Edges shape:", training_edges.shape)
print("Nodes shape:", training_node_features.shape)


<ipython-input-7-d4b50cdc2e0e>:64: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32


Edges shape: (2, 5020771)
Nodes shape: (2398933, 4)


In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((images, labels))

In [8]:
## GNN
gnn_model = GNNNodeClassifier(
    graph_info=graph_info,
    num_classes=2,
    hidden_units=hidden_units,
    dropout_rate=dropout_rate,
    name="gnn_model",
)

print("GNN output shape:", gnn_model([1, 10, 100]))

print(gnn_model.summary())

modelo = compile_model(gnn_model)

#train_data = ambos_features.iloc[:37943,:]
#test_data = ambos_features.iloc[37943:,:]

train_data = training_features.sample(frac=1)
test_data = validation_features.sample(frac=1)

print("Train data shape:", train_data.shape) 
print("Test data shape:", test_data.shape) 

# Create train and test features as a numpy array.
x_train = train_data[feature_names].to_numpy()
x_test = test_data[feature_names].to_numpy()
# Create train and test targets as a numpy array.
y_train = train_data["label"]
y_test = test_data["label"]


x_train = train_data.node.to_numpy()
print("x_train.dtype = ", x_train.dtype)
print("y_train.dtype = ", y_train.dtype)

#history = run_experiment(modelo, x_train, y_train)


GNN output shape: tf.Tensor(
[[-0.51589584 -0.16580537]
 [ 0.15424332 -0.05034565]
 [ 0.15424332 -0.05034565]], shape=(3, 2), dtype=float32)
Model: "gnn_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 preprocess (Sequential)     (2398933, 32)             1360      
                                                                 
 graph_conv1 (GraphConvLayer  multiple                 5888      
 )                                                               
                                                                 
 graph_conv2 (GraphConvLayer  multiple                 5888      
 )                                                               
                                                                 
 postprocess (Sequential)    (2398933, 32)             2368      
                                                                 
 logits (Dense)              multiple           

<ipython-input-8-610d830bb2e9>:26: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x_train = train_data[feature_names].to_numpy()
<ipython-input-8-610d830bb2e9>:27: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x_test = test_data[feature_names].to_numpy()


In [13]:
training_features.head()

,node,ID,OD,IDW,ODW,label
0,0,0,1,0,2239,0
1,252661,6,0,2907,0,0
2,1,1,1,6,6,0
3,97219,525631,540374,4796296,4936679,0
4,2,1,1,2,2,0


In [14]:
x_train

array([ 790171,  580089, 2119603, ..., 1217148, 2094851,  583581])

In [15]:
train_data.head()

,node,ID,OD,IDW,ODW,label
2016504,790171,1,0,2,0,0
534827,580089,1,1,1,1,0
1636645,2119603,1,1,1,1,0
411055,443333,1,1,3,3,0
1185943,1710239,1,1,1,1,0


In [16]:
y_train

2016504    0
534827     0
1636645    0
411055     0
1185943    0
          ..
1686292    0
694410     0
663064     0
1513545    0
538007     0
Name: label, Length: 2398933, dtype: int64

In [17]:
history = run_experiment(modelo, x_train, y_train)

Epoch 1/2


ResourceExhaustedError: Graph execution error:

Detected at node 'gnn_model/graph_conv1/sequential/dense_2/MatMul' defined at (most recent call last):
    File "/usr/lib/python3.8/runpy.py", line 194, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/usr/lib/python3.8/runpy.py", line 87, in _run_code
      exec(code, run_globals)
    File "/usr/local/lib/python3.8/dist-packages/ipykernel_launcher.py", line 16, in <module>
      app.launch_new_instance()
    File "/usr/local/lib/python3.8/dist-packages/traitlets/config/application.py", line 982, in launch_instance
      app.start()
    File "/usr/local/lib/python3.8/dist-packages/ipykernel/kernelapp.py", line 505, in start
      self.io_loop.start()
    File "/usr/local/lib/python3.8/dist-packages/tornado/platform/asyncio.py", line 215, in start
      self.asyncio_loop.run_forever()
    File "/usr/lib/python3.8/asyncio/base_events.py", line 570, in run_forever
      self._run_once()
    File "/usr/lib/py